# Lesson 04 ? Q-Learning in a Gridworld

Time for reinforcement learning! We'll teach an agent to navigate a tiny maze, balancing exploration with exploitation to maximize rewards.

## Learning Goals
- Understand the reinforcement learning loop (state ? action ? reward)
- Implement tabular Q-learning with ?-greedy exploration
- Visualize the learned policy and value function

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass

np.random.seed(21)
plt.style.use("seaborn-v0_8")

## Step 1 ? Define the Environment
Our grid has traps, a goal, and an energy penalty each step to incentivize fast solutions.

In [ ]:
ACTIONS = {
    0: np.array([0, 1]),   # right
    1: np.array([1, 0]),   # down
    2: np.array([0, -1]),  # left
    3: np.array([-1, 0])   # up
}

@dataclass
class Gridworld:
    width: int
    height: int
    start: tuple
    goal: tuple
    traps: tuple
    step_penalty: float = -0.04
    goal_reward: float = 1.0
    trap_penalty: float = -1.0

    def reset(self):
        self.agent_pos = np.array(self.start)
        return self._state_index(self.agent_pos)

    def _state_index(self, pos):
        return pos[0] * self.width + pos[1]

    def step(self, action):
        move = ACTIONS[action]
        next_pos = np.clip(self.agent_pos + move, [0, 0], [self.height - 1, self.width - 1])

        if tuple(next_pos) in self.traps:
            reward = self.trap_penalty
            done = True
        elif tuple(next_pos) == self.goal:
            reward = self.goal_reward
            done = True
        else:
            reward = self.step_penalty
            done = False

        self.agent_pos = next_pos
        return self._state_index(next_pos), reward, done


env = Gridworld(
    width=5,
    height=5,
    start=(4, 0),
    goal=(0, 4),
    traps=((1, 2), (2, 2), (3, 2))
)

state = env.reset()
state

## Step 2 ? Q-Learning Agent
We'll iteratively update a Q-table that estimates the value of taking an action in a state.

In [ ]:
num_states = env.width * env.height
num_actions = len(ACTIONS)

Q = np.zeros((num_states, num_actions))
alpha = 0.7      # learning rate
gamma = 0.95     # future reward discount
epsilon = 0.2    # exploration probability

def epsilon_greedy_policy(state, epsilon):
    if np.random.rand() < epsilon:
        return np.random.randint(num_actions)
    return np.argmax(Q[state])


episodes = 500
rewards_per_episode = []

for episode in range(episodes):
    state = env.reset()
    done = False
    total_reward = 0

    while not done:
        action = epsilon_greedy_policy(state, epsilon)
        next_state, reward, done = env.step(action)
        total_reward += reward

        best_next_action = np.argmax(Q[next_state])
        td_target = reward + gamma * Q[next_state, best_next_action] * (1 - done)
        td_error = td_target - Q[state, action]
        Q[state, action] += alpha * td_error

        state = next_state

    # decay exploration
    epsilon = max(0.05, epsilon * 0.995)
    rewards_per_episode.append(total_reward)

len(rewards_per_episode)

### Training Curve
We expect rewards to trend upward as the agent discovers shorter, safer routes.

In [ ]:
window = 20
running_avg = np.convolve(rewards_per_episode, np.ones(window) / window, mode="valid")

plt.figure(figsize=(7, 4))
plt.plot(running_avg)
plt.title("Smoothed Episode Rewards")
plt.xlabel("Episode")
plt.ylabel("Reward (20-episode moving average)")
plt.grid(True)
plt.show()

## Step 3 ? Visualize the Learned Policy
Let's convert the Q-table into human-readable arrows and highlight traps/goal.

In [ ]:
arrow_map = {0: "?", 1: "?", 2: "?", 3: "?"}
value_grid = Q.max(axis=1).reshape(env.height, env.width)
policy_grid = np.vectorize(lambda idx: arrow_map[idx])(Q.argmax(axis=1)).reshape(env.height, env.width)

for trap in env.traps:
    policy_grid[trap] = "?"
policy_grid[env.goal] = "?"
policy_grid[env.start] = "S"

print("Policy Grid (S = start, ? = goal, ? = trap):")
for row in policy_grid:
    print(" ".join(row))

plt.figure(figsize=(6, 5))
plt.imshow(value_grid, cmap="viridis")
plt.colorbar(label="State Value (max Q)")
plt.title("State Value Heatmap")
plt.show()

## Wrap-Up
Our agent learned to dodge traps and reach the goal through trial and error. Reinforcement learning scales from this toy example up to robotics and game-playing AIs?swap in larger state spaces, function approximators, or deep networks to level up.